In [1]:
from pyspark.sql.types import StructType, StructField, StringType, DecimalType, TimestampType
from pyspark.sql.functions import create_map, lit

In [2]:
dbutils.widgets.text("pipeline_id", "")
dbutils.widgets.text("run_id", "")
dbutils.widgets.text("task_id", "")
dbutils.widgets.text("processed_timestamp", "")
dbutils.widgets.text("catalog", "")

/mnt/c/Users/tariq/shah/dab_project/.venv/lib/python3.11/site-packages/databricks/sdk/_widgets/__init__.py:71: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(


In [3]:
pipeline_id = dbutils.widgets.get("pipeline_id")
run_id = dbutils.widgets.get("run_id")
task_id = dbutils.widgets.get("task_id")
processed_timestamp = dbutils.widgets.get("processed_timestamp")
catalog = dbutils.widgets.get("catalog")

In [4]:
schema = StructType([
    StructField("ride_id", StringType(), True),
    StructField("rideable_type", StringType(), True),
    StructField("started_at", TimestampType(), True),
    StructField("ended_at", TimestampType(), True),
    StructField("start_station_name", StringType(), True), 
    StructField("start_station_id", StringType(), True),   
    StructField("end_station_name", StringType(), True), 
    StructField("end_station_id", StringType(), True), 
    StructField("start_lat", DecimalType(), True), 
    StructField("start_lng", DecimalType(), True), 
    StructField("end_lat", DecimalType(), True), 
    StructField("end_lng", DecimalType(), True), 
    StructField("member_casual", StringType(), True), 
])

In [5]:
df = spark.read.csv(f"/Volumes/{catalog}/00_landing/source_citibike_data/JC-202503-citibike-tripdata.csv", schema=schema, header=True)

In [6]:
df = df.withColumn(
    "metadata",
    create_map(
        lit("pipeline_id"), lit(pipeline_id),
        lit("run_id"), lit(run_id),
        lit("task_id"), lit(task_id),
        lit("processed_timestamp"), lit(processed_timestamp)
    )
)

In [7]:
df.write.\
    mode("overwrite").\
    option("overwriteSchema", "true").\
    saveAsTable(f"{catalog}.01_bronze.jc_citibike")

AnalysisException: [UC_VOLUME_NOT_FOUND] Volume `00_landing`.`source_citibike_data`.`JC-202503-citibike-tripdata`.`csv` does not exist. Please use 'SHOW VOLUMES' to list available volumes. SQLSTATE: 42704

JVM stacktrace:
org.apache.spark.sql.catalyst.analysis.NoSuchVolumeException
	at com.databricks.managedcatalog.ManagedCatalogClientImpl$ConvertVolumeException.convertVolumeException(ManagedCatalogClientImpl.scala:7936)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.handleVolumeException(ManagedCatalogClientImpl.scala:7902)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.$anonfun$getVolume$1(ManagedCatalogClientImpl.scala:7656)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.$anonfun$recordAndWrapException$2(ManagedCatalogClientImpl.scala:6982)
	at com.databricks.spark.util.FrameProfiler$.record(FrameProfiler.scala:94)
	at com.databricks.managedcatalog.ManagedCatalogClientImpl.$anonfun$recordAndWrapException$1(ManagedCatalogClientImpl.scala:6981)
	at 